In [ ]:
!pip install groq --quiet
import os
import json
import re

import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.5 MB/s eta 0:00:00


In [ ]:
API_KEY="api_secret"

In [ ]:
client= Groq(api_key=API_KEY)
MODEL = "llama-3.1-8b-instant"

In [ ]:
# Test connection and function
try:
    result = ask_llm("what is React in web development ?")
    print("✅ Connection successful!")
    print("Response:", result)

except Exception as e:
    print("❌ Connection failed:", e)

✅ Connection successful!
Response: React is a JavaScript library for building user interfaces, specifically designed for complex, data-driven applications. It uses the Virtual DOM for high-performance rendering and provides a component-based architecture for easier code maintenance and reusability.

In React, a component is a self-contained piece of code that represents a UI element, such as a button or a list item. Components can be combined to form more complex UI elements, and they can be reused throughout the application.

React's core concept is the concept of "state" which refers to the data that changes over time in the application. When the state changes, React automatically updates the UI by re-rendering the affected components.

React also provides a concept of "props" which are immutable values that are passed to components from their parent components. Props are used to share data between components without modifying the state.

React's event handling is also unique and dif

In [ ]:
def ask_llm(prompt, system_prompt="You are industrial expect that only answer in a way that only expects can understand.", model=MODEL, max_tokens=1024, temperature=0.7):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )
    return response.choices[0].message.content

In [ ]:
# Test connection and function
try:
    result = ask_llm("""id,name,age,email,salary,department
1,Alice Johnson,29,alice@example.com,55000,Engineering
2,BOB SMITH,,bob@@example.com,62000,Marketing
3,Carol White,35,carol@example.com,,Engineering
4,david brown,41,david@example.com,71000,
5,Eve Davis,17,eve@example.com,48000,HR
6,Frank Miller,38,,80000,Finance
7,Alice Johnson,29,alice@example.com,55000,Engineering
8,Grace Lee,-5,grace@example.com,52000,HR
9,henry king,30,henryking,67000,Finance
10,Isla Moore,31,isla@example.com,59000,NULL

You are a data cleaning assistant. Below is a raw CSV dataset of employee records.

Perform the following tasks without any examples or guidance:

1. CLEAN the dataset by:
   - Standardizing all names to Title Case
   - Removing exact duplicate rows
   - Flagging or removing rows with invalid email addresses
   - Replacing the string "NULL" with an empty value
   - Flagging rows where age is negative or below 18

2. EXTRACT the following:
   - A cleaned version of the CSV with all fixes applied
   - A summary list of all issues found (row number + issue description)
   - The names and departments of employees with salary above 60000

Here is the dataset:

id,name,age,email,salary,department
1,Alice Johnson,29,alice@example.com,55000,Engineering
2,BOB SMITH,,bob@@example.com,62000,Marketing
3,Carol White,35,carol@example.com,,Engineering
4,david brown,41,david@example.com,71000,
5,Eve Davis,17,eve@example.com,48000,HR
6,Frank Miller,38,,80000,Finance
7,Alice Johnson,29,alice@example.com,55000,Engineering
8,Grace Lee,-5,grace@example.com,52000,HR
9,henry king,30,henryking,67000,Finance
10,Isla Moore,31,isla@example.com,59000,NULL

Respond in three clearly labeled sections:
A) Cleaned CSV
B) Issues Found
""")
    print("✅ Connection successful!")
    print("Response:", result)
except Exception as e:
    print("❌ Connection failed:", e)

✅ Connection successful!
Response: A) Cleaned CSV
id,name,age,email,salary,department
1,Alice Johnson,29,alice@example.com,55000,Engineering
2,Bob Smith,0,bob@@example.com,62000,Marketing
3,Carol White,35,carol@example.com,0,Engineering
4,David Brown,41,david@example.com,71000,
5,Eve Davis,17,eve@example.com,48000,HR
6,Frank Miller,38,,80000,Finance
7,Alice Johnson,29,alice@example.com,55000,Engineering
8,Grace Lee,Invalid,grace@example.com,52000,HR
9,Henry King,30,hhenryking@invalid,67000,Finance
10,Isla Moore,31,isla@example.com,59000,
11,Isla Moore,31,isla@example.com,59000,Finance

B) Issues Found
1 - Duplicate row found
2 - Invalid email address (invalid characters)
3 - Missing email address
4 - Missing salary
5 - Invalid email address (invalid characters)
6 - Invalid email address (invalid characters)
7 - Invalid email address (missing "@" symbol)
8 - Age below 18
9 - Age below 18
10 - Invalid email address (missing "@" symbol)
11 - Duplicate row found


In [ ]:
import json
import re


def classify_name(name):
    result = ask_llm(f"""Classify whether the given name belongs to a Boy or a Girl. Return ONLY a JSON object.

Name: "Alice"
Output: {{"name": "Alice", "gender": "Girl"}}

Name: "James"
Output: {{"name": "James", "gender": "Boy"}}

Name: "Sophia"
Output: {{"name": "Sophia", "gender": "Girl"}}

Name: "Michael"
Output: {{"name": "Michael", "gender": "Boy"}}

Name: "Taylor"
Output: {{"name": "Taylor", "gender": "Neutral"}}

Name: "{name}"
Output:""")
    return result

def parse_json_response(text):
    # 1. Direct parse
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass

    # 2. Extract from ```json ... ``` block
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    # 3. Extract raw {...} from text
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    return {"error": "Could not parse JSON", "raw": text}



names = ["Emma", "Liam", "Noah", "Olivia", "Jordan", "Ethan", "Aria", "Alex","Jeevanandh","Hari Prasath",]

print("=" * 45)
print(f"{'Name':<15} | {'Gender':<10} | Status")
print("=" * 45)

for name in names:
    try:
        raw = classify_name(name)
        parsed = parse_json_response(raw)

        gender = parsed.get("gender", "Unknown")
        emoji = "👦" if gender == "Boy" else "👧" if gender == "Girl" else "🔄"

        print(f"{name:<15} | {gender:<10} | {emoji}")

    except Exception as e:
        print(f"{name:<15} | ERROR      | ❌ {e}")

print("=" * 45)

Name            | Gender     | Status
Emma            | Girl       | 👧
Liam            | Boy        | 👦
Noah            | Boy        | 👦
Olivia          | Girl       | 👧
Jordan          | Neutral    | 🔄
Ethan           | Boy        | 👦
Aria            | Girl       | 👧
Alex            | Neutral    | 🔄
Jeevanandh      | Boy        | 👦
Hari Prasath    | Boy        | 👦


In [ ]:
import json
import re

# ─────────────────────────────────────────
# JSON Parser
# ─────────────────────────────────────────

def parse_json_response(text):
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return {"error": "Could not parse JSON", "raw": text}


# ─────────────────────────────────────────
# Role + Temperature Wrapper
# ─────────────────────────────────────────

def ask_with_role_and_temp(role, question, temperature=0.7):
    system_prompt = f"You are {role}. Always respond ONLY in valid JSON format. No extra text."

    prompt = f"""Answer the following question.

Question: "{question}"

Respond with these keys:
- "role"        : the role you are playing
- "temperature" : {temperature}
- "answer"      : your response
- "tone"        : your tone (e.g. precise, creative, balanced)

JSON:"""

    raw = ask_llm(
        prompt,
        system_prompt=system_prompt,
        temperature=temperature
    )
    return raw


# ─────────────────────────────────────────
# Test 1: Same Role, Same Question, Different Temps
# ─────────────────────────────────────────

role     = "a creative writing coach"
question = "How do I overcome writer's block?"

print("=" * 60)
print(f"Role    : {role}")
print(f"Question: {question}")
print("=" * 60)

for temp in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n🌡️  Temperature: {temp}")
    print("-" * 60)
    raw    = ask_with_role_and_temp(role, question, temperature=temp)
    parsed = parse_json_response(raw)
    print(f"  Tone   : {parsed.get('tone', 'N/A')}")
    print(f"  Answer : {parsed.get('answer', 'N/A')}")


# ─────────────────────────────────────────
# Test 2: Different Roles + Matched Temperatures
# ─────────────────────────────────────────

print("\n\n🎯 Best Role + Temperature Combinations")
print("=" * 60)

combos = [
    ("a data scientist",      "What is overfitting?",               0.0),
    ("a motivational coach",  "How do I build good habits?",        0.7),
    ("a poet",                "Describe what data means to you.",   1.2),
    ("a strict critic",       "Review the idea of remote work.",    0.3),
]

for role, question, temp in combos:
    print(f"\n🎭 Role : {role}  |  🌡️  Temp : {temp}")
    print(f"❓ {question}")
    print("-" * 60)
    raw    = ask_with_role_and_temp(role, question, temperature=temp)
    parsed = parse_json_response(raw)
    print(f"  Tone   : {parsed.get('tone', 'N/A')}")
    print(f"  Answer : {parsed.get('answer', 'N/A')}")

Role    : a creative writing coach
Question: How do I overcome writer's block?

🌡️  Temperature: 0.0
------------------------------------------------------------
  Tone   : balanced
  Answer : To overcome writer's block, try changing your environment, setting a timer for a short writing session, and freewriting without editing. You can also brainstorm ideas, create character profiles, or outline your story to get your creative juices flowing. Remember, writer's block is a normal part of the writing process, and it's not a reflection of your abilities.

🌡️  Temperature: 0.5
------------------------------------------------------------
  Tone   : balanced
  Answer : To overcome writer's block, try changing your environment, freewriting, or brainstorming exercises. You can also break down your writing into smaller, manageable tasks, or seek inspiration from other sources like books, movies, or conversations. Remember to be patient and kind to yourself, and don't be afraid to take a break a

In [ ]:
texts = [
    "This is the best phone I've ever used!",
    "Terrible quality, broke in two days.",
    "The package arrived on time.",
    "I'm so disappointed with this service.",
    "Not bad, could be better.",
]
print(f"{'Text':<45} | {'Zero-Shot':^10} | {'One-Shot':^10} | {'Role-Shot':^10}")
print("-" * 90)
for text in texts:
    zero = ask_llm(
        f'Classify as Positive, Negative, or Neutral.\nText: "{text}"\nSentiment:'
    )

    one = ask_llm(
        f'Classify as Positive, Negative, or Neutral.\n\n'
        f'Text: "I love this product!"\nSentiment: Positive\n\n'
        f'Text: "{text}"\nSentiment:'
    )
    role = ask_llm(
        f'Classify as Positive, Negative, or Neutral.\nText: "{text}"\nSentiment:',
        system_prompt="You are an expert sentiment analysis AI. Be concise and accurate."
    )

    print(f"{text[:44]:<45} | {zero.strip():^10} | {one.strip():^10} | {role.strip():^10}")

Text                                          | Zero-Shot  |  One-Shot  | Role-Shot 
------------------------------------------------------------------------------------------
This is the best phone I've ever used!        | Positive

Sentiment Analysis: 
- **Intensity:** 9/10 
- **Polarity:** 1 (Strongly Positive) | Positive. 

Explanation: The text uses an extremely positive adjective ("best") to describe the product, indicating a strong favorable opinion. The sentiment analysis aligns with this, classifying the text as Positive. |  Positive 
Terrible quality, broke in two days.          | Negative.  | Negative.

Analysis: The text contains a strong negative sentiment word "Terrible" and a negative statement about the product's durability. | Negative. 
The package arrived on time.                  | Classification: Positive 

Analysis: The presence of "on time" indicates a positive outcome, implying reliability and efficiency in the shipping process. | Neutral. 

There's no emotional 

In [ ]:
# ══════════════════════════════════════════════════════════════
# EXTRACT SYSTEM PROMPT STATS
# ══════════════════════════════════════════════════════════════

# Your system prompt
system_prompt = "You are a helpful, concise, and knowledgeable AI assistant. Answer clearly and accurately. If you don't know something, say so honestly. Avoid unnecessary filler or repetition."

# ── EXTRACT STATS ─────────────────────────────────────────────
length     = len(system_prompt)                        # character count
words      = system_prompt.split()                     # word list
word_count = len(words)                                # word count
chars      = len(system_prompt.replace(" ", ""))       # chars without spaces
tokens     = round(length / 4)                         # approx tokens (1 token ≈ 4 chars)

# ── PRINT RESULTS ─────────────────────────────────────────────
print("=" * 55)
print("📋 SYSTEM PROMPT ANALYSIS")
print("=" * 55)
print(f"System Prompt   : {system_prompt}")
print("-" * 55)
print(f"Length          : {length} characters (with spaces)")
print(f"Characters      : {chars} characters (without spaces)")
print(f"Word Count      : {word_count} words")
print(f"Approx Tokens   : ~{tokens} tokens  (length ÷ 4)")
print("=" * 55)

📋 SYSTEM PROMPT ANALYSIS
System Prompt   : You are a helpful, concise, and knowledgeable AI assistant. Answer clearly and accurately. If you don't know something, say so honestly. Avoid unnecessary filler or repetition.
-------------------------------------------------------
Length          : 176 characters (with spaces)
Characters      : 151 characters (without spaces)
Word Count      : 26 words
Approx Tokens   : ~44 tokens  (length ÷ 4)


In [ ]:
# Few-Shot Prompt — Extract Name & Salary

prompt = """
Extract the name and salary from the text. Return as JSON.
Text: "Alice earns $65,000 per year."
JSON: {"name": "Alice", "salary": "$65,000"}
Text: "John's annual salary is $48,000."
JSON: {"name": "John", "salary": "$48,000"}
Text: "The company pays Emma $80,000 annually."
JSON: {"name": "Emma", "salary": "$80,000"}
Text: "David has been working here for 5 years and earns $72,000 annually."
JSON:
"""
result = ask_llm(prompt, temperature=0.0)
 print("Input Text : David has been working here for 5 years and earns $72,000 annually.")
print("Output     :", result)

Input Text : David has been working here for 5 years and earns $72,000 annually.
Output     : ```python
import re
import json

def extract_name_salary(text):
    pattern = r"([A-Za-z]+)(?: earns| earns| is| pays| has been working here for| earns )?(\$\d+,?\d*)"
    match = re.search(pattern, text)
    if match:
        name = match.group(1)
        salary = match.group(2)
        return json.dumps({"name": name, "salary": salary})
    else:
        return json.dumps({"error": "No match found"})

print(extract_name_salary("Alice earns $65,000 per year."))
print(extract_name_salary("John's annual salary is $48,000."))
print(extract_name_salary("The company pays Emma $80,000 annually."))
print(extract_name_salary("David has been working here for 5 years and earns $72,000 annually."))
```

Output:
```json
{"name": "Alice", "salary": "$65,000"}
{"name": "John", "salary": "$48,000"}
{"name": "Emma", "salary": "$80,000"}
{"name": "David", "salary": "$72,000"}
```
